In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("dq_flags.csv")

# -------------------- FEATURE ENGINEERING --------------------

df_fe = df.copy()

# Numeric features
numeric_features = []

# Claim amount (robust transforms)
if "ClaimAmount_NUM" in df_fe.columns:
    df_fe["claim_log"] = np.log1p(df_fe["ClaimAmount_NUM"].clip(lower=0.01))
    numeric_features.append("claim_log")

# Temporal features
if "ClaimDate_PARSED" in df_fe.columns and "ServiceDate_PARSED" in df_fe.columns:
    df_fe["days_between"] = (
        (pd.to_datetime(df_fe["ClaimDate_PARSED"]) - 
         pd.to_datetime(df_fe["ServiceDate_PARSED"])).dt.days
    )
    numeric_features.append("days_between")

if "DOB_PARSED" in df_fe.columns and "ClaimDate_PARSED" in df_fe.columns:
    df_fe["age"] = (
        (pd.to_datetime(df_fe["ClaimDate_PARSED"]) - 
         pd.to_datetime(df_fe["DOB_PARSED"])).dt.days / 365.25
    )
    numeric_features.append("age")

# Provider & patient stats
if "ProviderID" in df_fe.columns:
    df_fe["provider_claim_count"] = df_fe.groupby("ProviderID")["ClaimID"].transform("count")
    numeric_features.append("provider_claim_count")

if "PatientID" in df_fe.columns:
    df_fe["patient_claim_count"] = df_fe.groupby("PatientID")["ClaimID"].transform("count")
    numeric_features.append("patient_claim_count")

# DQ flags (already numeric)
flag_cols = [
    c for c in df_fe.columns 
    if ("bad_" in c) or ("flag" in c.lower()) or c in [
        "missing_key","dup_claimid_flag","dup_full_row","amount_suspect_flag","dq_score"
    ]
]
numeric_features.extend(flag_cols)

# Categorical features
categorical_features = ["PatientGender","ClaimStatus","ClaimType",
                        "ProviderSpecialty","State"]
categorical_features = [c for c in categorical_features if c in df_fe.columns]

# -------------------- ENCODING PIPELINE --------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X = preprocessor.fit_transform(df_fe)

# Save feature matrix
X_df = pd.DataFrame(X.toarray() if hasattr(X, "toarray") else X)
X_df.to_csv("claims_features_scaled.csv", index=False)

print("Feature engineering complete.")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Final feature matrix shape:", X_df.shape)


Feature engineering complete.
Numeric features: 18
Categorical features: 5
Final feature matrix shape: (3090, 99)


In [ ]:
# =========================== STAGE 4: UNSUPERVISED ANOMALY DETECTION ===========================
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler

# Optional: HBOS (pyOD)
try:
    from pyod.models.hbos import HBOS
    HAS_HBOS = True
except:
    HAS_HBOS = False

# -------------------------------- CONFIG --------------------------------
FEATS_FILE = "claims_features_scaled.csv"
DQ_FILE    = "dq_flags.csv"
RAW_FILE   = "payer_unsupervised_anomaly_data.csv"
OUT_FILE   = "anomaly_scores.csv"

CONTAMINATION = 0.04   # Based on your injected anomaly prevalence
RANDOM_STATE = 42

# Default ensemble weights
WEIGHTS = {
    "if": 0.5,
    "lof": 0.35,
    "hbos": 0.15    # auto-adjust if HBOS not installed
}
# -------------------------------------------------------------------------


# =========================== LOAD THE FEATURE MATRIX =============================
if not os.path.exists(FEATS_FILE):
    raise FileNotFoundError(f"{FEATS_FILE} not found! Run Feature Engineering first.")

X = pd.read_csv(FEATS_FILE)
n = len(X)
print(f"Loaded {n} ML feature rows | Feature Dim: {X.shape[1]}")

# Load additional context (optional)
dq  = pd.read_csv(DQ_FILE) if os.path.exists(DQ_FILE) else None
raw = pd.read_csv(RAW_FILE) if os.path.exists(RAW_FILE) else None


# =========================== FIX: HANDLE NAN FOR LOF =============================
# LOF *cannot* handle NaN → use median-imputation ONLY for ML matrix
X_filled = X.fillna(X.median())


# =========================== MODEL 1: ISOLATION FOREST ===========================
if_model = IsolationForest(
    n_estimators=300,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE
)
if_model.fit(X_filled)

if_score_raw = -if_model.decision_function(X_filled)
if_flag = (if_model.predict(X_filled) == -1).astype(int)


# =========================== MODEL 2: LOCAL OUTLIER FACTOR ===========================
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    novelty=False
)
lof_pred = lof.fit_predict(X_filled)
lof_score_raw = -lof.negative_outlier_factor_
lof_flag = (lof_pred == -1).astype(int)


# =========================== MODEL 3: HBOS ===========================
if HAS_HBOS:
    hbos = HBOS(contamination=CONTAMINATION)
    hbos.fit(X_filled)
    hbos_score_raw = hbos.decision_scores_
    hbos_flag = hbos.predict(X_filled)
else:
    hbos_score_raw = np.zeros(n)
    hbos_flag = np.zeros(n, dtype=int)
    
    WEIGHTS = {"if": 0.6, "lof": 0.4, "hbos": 0.0}


# =========================== NORMALIZE ALL MODEL SCORES ===========================
scores = pd.DataFrame({
    "if_raw":  if_score_raw,
    "lof_raw": lof_score_raw,
    "hbos_raw": hbos_score_raw
})

scaler = MinMaxScaler()
scores_norm = pd.DataFrame(
    scaler.fit_transform(scores),
    columns=scores.columns
)


# =========================== WEIGHTED SCORE FUSION ===========================
total_w = WEIGHTS["if"] + WEIGHTS["lof"] + WEIGHTS["hbos"]
w_if   = WEIGHTS["if"]  / total_w
w_lof  = WEIGHTS["lof"] / total_w
w_hbos = WEIGHTS["hbos"] / total_w

combined_score = (
    scores_norm["if_raw"]  * w_if +
    scores_norm["lof_raw"] * w_lof +
    scores_norm["hbos_raw"]* w_hbos
)


# =========================== FINAL ANOMALY FLAG ===========================
threshold = np.quantile(combined_score, 1 - CONTAMINATION)
final_flag = (combined_score >= threshold).astype(int)

print("\n================ RESULTS ================")
print("Threshold used:", threshold)
print(f"Anomalies flagged: {final_flag.sum()} / {n}  ({final_flag.mean()*100:.2f}%)")
print("==========================================\n")


# =========================== BUILD OUTPUT TABLE ===========================
if raw is not None and len(raw) == n:
    out = raw.copy()
else:
    out = pd.DataFrame({"row_index": np.arange(n)})

# Add ML scores and flags
out["if_score_raw"] = if_score_raw
out["lof_score_raw"] = lof_score_raw
out["hbos_score_raw"] = hbos_score_raw

out["if_flag"]  = if_flag
out["lof_flag"] = lof_flag
out["hbos_flag"] = hbos_flag

out["if_score_norm"] = scores_norm["if_raw"]
out["lof_score_norm"] = scores_norm["lof_raw"]
out["hbos_score_norm"] = scores_norm["hbos_raw"]

out["combined_score"] = combined_score
out["final_anomaly_flag"] = final_flag

# Merge DQ flags
if dq is not None and len(dq) == n:
    out = pd.concat([out, dq], axis=1)

# Save results
out.to_csv(OUT_FILE, index=False)
print(f"Saved: {OUT_FILE}")


# =========================== SHOW TOP 10 ANOMALIES ===========================
print("\nTop 10 anomalies:")
top = out.sort_values("combined_score", ascending=False).head(10)
print(top[["combined_score", "final_anomaly_flag"]].to_string(index=False))


Loaded 3090 ML feature rows | Feature Dim: 99

================ RESULTS ================
Threshold used: 0.5119975734150799
Anomalies flagged: 124 / 3090  (4.01%)

Saved: anomaly_scores.csv

Top 10 anomalies:
 combined_score  final_anomaly_flag
       0.899556                   1
       0.780450                   1
       0.779549                   1
       0.751159                   1
       0.723044                   1
       0.711747                   1
       0.696323                   1
       0.651945                   1
       0.650443                   1
       0.649302                   1


In [5]:
# ===================== STAGE 5: FINAL SEVERITY SCORING =====================
import pandas as pd
import numpy as np
import json
from datetime import datetime
import os

IN_FILE  = "anomaly_scores.csv"        # from Stage 4
OUT_FILE = "final_anomaly_severity.csv"
OUT_SUMMARY = "severity_summary.json"

if not os.path.exists(IN_FILE):
    raise FileNotFoundError(f"{IN_FILE} not found. Run Stage 4 first.")

df = pd.read_csv(IN_FILE)
n = len(df)
print(f"Loaded {n} rows from {IN_FILE}")

# -------------------- CHECK REQUIRED COLUMNS --------------------
# ML score & flag
if "combined_score" not in df.columns:
    raise ValueError("combined_score missing in input. Check Stage 4 output.")
if "final_anomaly_flag" not in df.columns:
    raise ValueError("final_anomaly_flag missing in input. Check Stage 4 output.")

# DQ score (optional but expected)
if "dq_score" not in df.columns:
    print("WARNING: dq_score missing. Final severity will use ML score only.")
    df["dq_score"] = 0.0

# Ensure both are numeric
df["combined_score"] = pd.to_numeric(df["combined_score"], errors="coerce").fillna(0.0)
df["dq_score"] = pd.to_numeric(df["dq_score"], errors="coerce").fillna(0.0)

# -------------------- NORMALIZE INPUT SCORES (SAFETY) --------------------
# combined_score is already 0–1, dq_score is 0–1, but we enforce bounds
df["combined_score"] = df["combined_score"].clip(lower=0.0, upper=1.0)
df["dq_score"] = df["dq_score"].clip(lower=0.0, upper=1.0)

# -------------------- FINAL SEVERITY SCORE --------------------
# Idea:
#   - ML anomaly score (combined_score) captures statistical weirdness
#   - dq_score captures data quality issues
#   - We blend them into one severity score in [0, 1]
#
# Weight more on ML, but still value DQ:
#   final_severity_score = 0.6 * ML + 0.4 * DQ

ALPHA_ML = 0.6  # ML weight
ALPHA_DQ = 0.4  # DQ weight

df["final_severity_score"] = ALPHA_ML * df["combined_score"] + ALPHA_DQ * df["dq_score"]

# Extra rule: if final_anomaly_flag==0 but dq_score is very high, boost severity
# e.g., rows with dq_score >= 0.7 are serious DQ failures even if ML didn't flag them.
high_dq_mask = (df["final_anomaly_flag"] == 0) & (df["dq_score"] >= 0.7)
df.loc[high_dq_mask, "final_severity_score"] = np.maximum(
    df.loc[high_dq_mask, "final_severity_score"],
    0.7
)

# -------------------- MAP SCORE TO SEVERITY LABEL --------------------
# Clear, interpretable buckets:
#   [0.80, 1.00] -> CRITICAL
#   [0.60, 0.80) -> HIGH
#   [0.40, 0.60) -> MEDIUM
#   (0.00, 0.40) -> LOW
#   == 0.00       -> CLEAN (no ML anomaly, no DQ problems)

def severity_label(score):
    if score >= 0.80:
        return "CRITICAL"
    elif score >= 0.60:
        return "HIGH"
    elif score >= 0.40:
        return "MEDIUM"
    elif score > 0.00:
        return "LOW"
    else:
        return "CLEAN"

df["final_severity_label"] = df["final_severity_score"].apply(severity_label)

# For consistency: if final_anomaly_flag==1 but severity ended up LOW/CLEAN,
# lift it to at least MEDIUM (ML saw it as anomaly)
mask_anom_but_low = (df["final_anomaly_flag"] == 1) & df["final_severity_score"] < 0.40
df.loc[mask_anom_but_low, "final_severity_score"] = np.maximum(
    df.loc[mask_anom_but_low, "final_severity_score"],
    0.45
)
df.loc[mask_anom_but_low, "final_severity_label"] = "MEDIUM"

# -------------------- SAVE FINAL TABLE --------------------
df.to_csv(OUT_FILE, index=False)
print(f"\nSaved final severity table to: {OUT_FILE}")

# -------------------- BUILD SUMMARY FOR REPORTING --------------------
severity_counts = df["final_severity_label"].value_counts().to_dict()
ml_flag_rate = float(df["final_anomaly_flag"].mean())
dq_issue_rate = float((df["dq_score"] > 0).mean())

summary = {
    "rows": int(n),
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
    "severity_counts": severity_counts,
    "ml_flag_rate": ml_flag_rate,
    "dq_issue_rate": dq_issue_rate,
    "avg_final_severity_score": float(df["final_severity_score"].mean()),
}

with open(OUT_SUMMARY, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSeverity summary:")
for level in ["CRITICAL","HIGH","MEDIUM","LOW","CLEAN"]:
    if level in severity_counts:
        print(f"  {level:8s}: {severity_counts[level]}")

print(f"\nML anomaly rate (final_anomaly_flag=1): {ml_flag_rate*100:.2f}%")
print(f"Rows with any DQ issue (dq_score>0):      {dq_issue_rate*100:.2f}%")
print(f"Average final severity score:            {summary['avg_final_severity_score']:.3f}")

print(f"\nSaved JSON summary to: {OUT_SUMMARY}")


Loaded 3090 rows from anomaly_scores.csv

Saved final severity table to: final_anomaly_severity.csv

Severity summary:
  HIGH    : 5
  MEDIUM  : 3062
  LOW     : 23

ML anomaly rate (final_anomaly_flag=1): 4.01%
Rows with any DQ issue (dq_score>0):      98.12%
Average final severity score:            0.450

Saved JSON summary to: severity_summary.json
